# 04 - PD model: logistic regression scorecard

Logistic regression on the WoE values of the 20 short-listed variables, fitted on the training set. Coefficient signs are checked for business logic before the model is turned into a scorecard.

In [25]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from src.woe import load_binners, woe_transform

train = pd.read_parquet("../data/processed/train.parquet")
test = pd.read_parquet("../data/processed/test.parquet")
binners = load_binners("../models/binners.joblib")

X_train = woe_transform(train, binners)
X_test = woe_transform(test, binners)
y_train, y_test = train["TARGET"], test["TARGET"]

##model = LogisticRegression(penalty=None, max_iter=1000)
model = LogisticRegression(C=np.inf, max_iter=1000)
model.fit(X_train, y_train)

coefs = pd.Series(model.coef_[0], index=X_train.columns).sort_values()
print(coefs.round(3))

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"\nTest AUC: {auc:.3f}, Gini: {2 * auc - 1:.3f}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
EXT_SOURCE_3                  -0.819
EXT_SOURCE_2                  -0.733
CREDIT_TERM                   -0.655
OWN_CAR_AGE                   -0.576
ORGANIZATION_TYPE             -0.546
NAME_EDUCATION_TYPE           -0.513
EXT_SOURCE_1                  -0.512
CREDIT_GOODS_RATIO            -0.497
AMT_GOODS_PRICE               -0.434
DAYS_EMPLOYED                 -0.393
DAYS_REGISTRATION             -0.328
OCCUPATION_TYPE               -0.324
REGION_RATING_CLIENT_W_CITY   -0.314
AMT_ANNUITY                   -0.288
DAYS_ID_PUBLISH               -0.279
REG_CITY_NOT_LIVE_CITY        -0.276
NAME_FAMILY_STATUS            -0.239
DAYS_LAST_PHONE_CHANGE        -0.168
REG_CITY_NOT_WORK_CITY        -0.027
AGE_YEARS                      0.142
dtype: float64

Test AUC: 0.746, Gini: 0.493


## remove AGE_YEARS and refit

AGE_YEARS is the only coefficient that gives positive AUC, we exclude it and refit the model 

In [26]:
drop = ["AGE_YEARS"]
cols = [c for c in X_train.columns if c not in drop]

model = LogisticRegression(C=np.inf, max_iter=1000)
model.fit(X_train[cols], y_train)

coefs = pd.Series(model.coef_[0], index=cols).sort_values()
print(coefs.round(3))

auc = roc_auc_score(y_test, model.predict_proba(X_test[cols])[:, 1])
print(f"\nTest AUC: {auc:.3f}, Gini: {2 * auc - 1:.3f}")

EXT_SOURCE_3                  -0.816
EXT_SOURCE_2                  -0.731
CREDIT_TERM                   -0.659
OWN_CAR_AGE                   -0.586
NAME_EDUCATION_TYPE           -0.536
CREDIT_GOODS_RATIO            -0.497
ORGANIZATION_TYPE             -0.496
EXT_SOURCE_1                  -0.490
AMT_GOODS_PRICE               -0.427
DAYS_EMPLOYED                 -0.374
OCCUPATION_TYPE               -0.338
REGION_RATING_CLIENT_W_CITY   -0.315
AMT_ANNUITY                   -0.291
DAYS_REGISTRATION             -0.283
REG_CITY_NOT_LIVE_CITY        -0.263
DAYS_ID_PUBLISH               -0.234
NAME_FAMILY_STATUS            -0.208
DAYS_LAST_PHONE_CHANGE        -0.166
REG_CITY_NOT_WORK_CITY        -0.009
dtype: float64

Test AUC: 0.746, Gini: 0.493


## Remove variables with wrong or zero effect, and refit

- **AGE_YEARS** had a positive coefficient (+0.142): in combination with the other variables, its effect reversed. A wrong-sign coefficient cannot be explained, so it is removed. Test Gini is unchanged (0.493), confirming that other variables, such as DAYS_EMPLOYED and ORGANIZATION_TYPE, already carry its information.
- **REG_CITY_NOT_WORK_CITY** had a coefficient close to zero (−0.009) once AGE_YEARS was removed, so it adds nothing beyond REG_CITY_NOT_LIVE_CITY. It is removed.

After refitting, all 18 coefficients are negative, as expected with WoE values where positive means lower risk. Test Gini remains 0.493.

In [27]:
drop = ["REG_CITY_NOT_WORK_CITY", "AGE_YEARS"]
cols = [c for c in X_train.columns if c not in drop]

model = LogisticRegression(C=np.inf, max_iter=1000)
model.fit(X_train[cols], y_train)

coefs = pd.Series(model.coef_[0], index=cols).sort_values()
print(coefs.round(3))

auc = roc_auc_score(y_test, model.predict_proba(X_test[cols])[:, 1])
print(f"\nTest AUC: {auc:.3f}, Gini: {2 * auc - 1:.3f}")

EXT_SOURCE_3                  -0.816
EXT_SOURCE_2                  -0.732
CREDIT_TERM                   -0.653
OWN_CAR_AGE                   -0.579
NAME_EDUCATION_TYPE           -0.536
CREDIT_GOODS_RATIO            -0.499
ORGANIZATION_TYPE             -0.496
EXT_SOURCE_1                  -0.490
AMT_GOODS_PRICE               -0.426
DAYS_EMPLOYED                 -0.373
OCCUPATION_TYPE               -0.339
REGION_RATING_CLIENT_W_CITY   -0.314
AMT_ANNUITY                   -0.290
REG_CITY_NOT_LIVE_CITY        -0.289
DAYS_REGISTRATION             -0.282
DAYS_ID_PUBLISH               -0.233
NAME_FAMILY_STATUS            -0.205
DAYS_LAST_PHONE_CHANGE        -0.165
dtype: float64

Test AUC: 0.746, Gini: 0.493


## Backward elimination with cross-validation

Aiming to about 8–12 variables, each adding real value in combination with the others. Each variable is removed in turn, and the drop in Gini is measured. Variables whose removal costs almost nothing are candidates for removal.

Selection uses 5-fold cross-validation on the training set only. The test set is kept aside for the final, unbiased evaluation of the chosen model.

In [28]:
from sklearn.model_selection import cross_val_score

def cv_gini(cols):
    model = LogisticRegression(C=np.inf, max_iter=1000)
    auc = cross_val_score(model, X_train[cols], y_train, cv=5, scoring="roc_auc").mean()
    return 2 * auc - 1

cols = [c for c in X_train.columns if c not in ["AGE_YEARS", "REG_CITY_NOT_WORK_CITY"]]
base = cv_gini(cols)
print(f"Base CV Gini ({len(cols)} variables): {base:.4f}")

drop_effect = pd.Series({
    var: base - cv_gini([c for c in cols if c != var]) for var in cols
}).sort_values()
print(drop_effect.round(4))

Base CV Gini (18 variables): 0.4935
AMT_ANNUITY                    0.0002
NAME_FAMILY_STATUS             0.0002
DAYS_LAST_PHONE_CHANGE         0.0003
DAYS_ID_PUBLISH                0.0005
REG_CITY_NOT_LIVE_CITY         0.0005
DAYS_REGISTRATION              0.0005
REGION_RATING_CLIENT_W_CITY    0.0009
OCCUPATION_TYPE                0.0013
OWN_CAR_AGE                    0.0017
NAME_EDUCATION_TYPE            0.0026
CREDIT_TERM                    0.0027
DAYS_EMPLOYED                  0.0028
AMT_GOODS_PRICE                0.0029
ORGANIZATION_TYPE              0.0037
CREDIT_GOODS_RATIO             0.0041
EXT_SOURCE_1                   0.0099
EXT_SOURCE_2                   0.0376
EXT_SOURCE_3                   0.0601
dtype: float64


### Iterative backward elimination

Removing several weak variables at once can cost more than the sum of their individual effects, because they may partly stand in for each other. Variables are therefore removed one at a time: in each round, the variable whose removal costs the least Gini is dropped, and the effects are re-measured. The process stops when every remaining variable costs at least 0.001 in cross-validated Gini when removed.

In [29]:
THRESHOLD = 0.001
selected = cols.copy()
log = []

while True:
    base = cv_gini(selected)
    effects = pd.Series({
        var: base - cv_gini([c for c in selected if c != var]) for var in selected
    })
    weakest = effects.idxmin()
    if effects.min() >= THRESHOLD:
        break
    selected.remove(weakest)
    log.append({"removed": weakest, "gini_drop": effects.min(), "gini_after": cv_gini(selected)})

print(pd.DataFrame(log).round(4))
print(f"\nFinal: {len(selected)} variables, CV Gini {cv_gini(selected):.4f}")
print(selected)

                  removed  gini_drop  gini_after
0             AMT_ANNUITY     0.0002      0.4933
1      NAME_FAMILY_STATUS     0.0002      0.4931
2  DAYS_LAST_PHONE_CHANGE     0.0003      0.4929
3       DAYS_REGISTRATION     0.0005      0.4924
4  REG_CITY_NOT_LIVE_CITY     0.0006      0.4918
5         DAYS_ID_PUBLISH     0.0007      0.4912

Final: 12 variables, CV Gini 0.4912
['EXT_SOURCE_3', 'EXT_SOURCE_2', 'EXT_SOURCE_1', 'DAYS_EMPLOYED', 'AMT_GOODS_PRICE', 'OCCUPATION_TYPE', 'CREDIT_GOODS_RATIO', 'ORGANIZATION_TYPE', 'REGION_RATING_CLIENT_W_CITY', 'NAME_EDUCATION_TYPE', 'CREDIT_TERM', 'OWN_CAR_AGE']


**Result:** six variables were removed, reducing cross-validated Gini by only 0.0023 (from 0.4935 to 0.4912).

| Removed | Gini drop | Gini after |
|---|---|---|
| AMT_ANNUITY | 0.0002 | 0.4933 |
| NAME_FAMILY_STATUS | 0.0002 | 0.4931 |
| DAYS_LAST_PHONE_CHANGE | 0.0003 | 0.4929 |
| DAYS_REGISTRATION | 0.0005 | 0.4924 |
| REG_CITY_NOT_LIVE_CITY | 0.0006 | 0.4918 |
| DAYS_ID_PUBLISH | 0.0007 | 0.4912 |

The final model has 12 variables: the three external scores, employment (DAYS_EMPLOYED, OCCUPATION_TYPE, ORGANIZATION_TYPE), education, region, loan structure (AMT_GOODS_PRICE, CREDIT_GOODS_RATIO, CREDIT_TERM) and car age.

NAME_FAMILY_STATUS, kept provisionally after the business review, adds almost nothing in combination and is removed. This also removes a sensitive characteristic from the model. The stability variables added after EDA (phone change, ID document, registration and address mismatch) were each weakly predictive alone, but add little once the other variables are included.

## refit the model on the full training set with the 12 variables, check that all coefficients are negative, and measure the test Gini once.

In [30]:
FINAL_VARS = [
    "EXT_SOURCE_3", "EXT_SOURCE_2", "EXT_SOURCE_1",
    "DAYS_EMPLOYED", "OCCUPATION_TYPE", "ORGANIZATION_TYPE",
    "NAME_EDUCATION_TYPE", "REGION_RATING_CLIENT_W_CITY",
    "AMT_GOODS_PRICE", "CREDIT_GOODS_RATIO", "CREDIT_TERM",
    "OWN_CAR_AGE",
]

print(FINAL_VARS)
print(len(FINAL_VARS))

['EXT_SOURCE_3', 'EXT_SOURCE_2', 'EXT_SOURCE_1', 'DAYS_EMPLOYED', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_EDUCATION_TYPE', 'REGION_RATING_CLIENT_W_CITY', 'AMT_GOODS_PRICE', 'CREDIT_GOODS_RATIO', 'CREDIT_TERM', 'OWN_CAR_AGE']
12


In [31]:
model = LogisticRegression(C=np.inf, max_iter=1000)
model.fit(X_train[FINAL_VARS], y_train)

coefs = pd.Series(model.coef_[0], index=FINAL_VARS).sort_values()
print(coefs.round(3))

auc = roc_auc_score(y_test, model.predict_proba(X_test[FINAL_VARS])[:, 1])
print(f"\nTest AUC: {auc:.3f}, Gini: {2 * auc - 1:.3f}")

EXT_SOURCE_3                  -0.833
EXT_SOURCE_2                  -0.752
CREDIT_TERM                   -0.664
OWN_CAR_AGE                   -0.585
ORGANIZATION_TYPE             -0.567
EXT_SOURCE_1                  -0.523
CREDIT_GOODS_RATIO            -0.523
NAME_EDUCATION_TYPE           -0.503
AMT_GOODS_PRICE               -0.493
DAYS_EMPLOYED                 -0.448
REGION_RATING_CLIENT_W_CITY   -0.329
OCCUPATION_TYPE               -0.328
dtype: float64

Test AUC: 0.745, Gini: 0.490


## Turn the model into a scorecard

A scorecard gives points per bin, and a client's total score is the sum of their points. Higher scores mean lower risk, which makes the model easy to explain to business users.

Scaling: 600 points correspond to good-to-bad odds of 50:1, and every 20 points (PDO, points to double the odds) doubles the odds.

### We start with looking at the variable of EXT_SOURCE_3

In [32]:
PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50
factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)
alpha = model.intercept_[0]
n = len(FINAL_VARS)

rows = []
for var, beta in zip(FINAL_VARS, model.coef_[0]):
    table = binners[var].binning_table.build().iloc[:-1]   # drop the Totals row
    for _, r in table.iterrows():
        woe = float(r["WoE"])
        points = (offset - factor * alpha) / n - factor * beta * woe
        rows.append({"variable": var, "bin": r["Bin"], "woe": woe, "points": round(points)})

scorecard = pd.DataFrame(rows)
scorecard[scorecard["variable"] == "EXT_SOURCE_3"]

,variable,bin,woe,points
0,EXT_SOURCE_3,"(-inf, 0.18)",-1.156807,19
1,EXT_SOURCE_3,"[0.18, 0.31)",-0.631727,31
2,EXT_SOURCE_3,"[0.31, 0.41)",-0.224796,41
3,EXT_SOURCE_3,"[0.41, 0.50)",0.090854,49
4,EXT_SOURCE_3,"[0.50, 0.57)",0.338730,55
5,EXT_SOURCE_3,"[0.57, 0.64)",0.544172,59
6,EXT_SOURCE_3,"[0.64, 0.70)",0.741244,64
7,EXT_SOURCE_3,"[0.70, inf)",0.940644,69
8,EXT_SOURCE_3,Special,0.000000,46
9,EXT_SOURCE_3,Missing,-0.160170,43


### Check the whole scorecard

In [33]:
ranges = scorecard.groupby("variable")["points"].agg(["min", "max"])
ranges["range"] = ranges["max"] - ranges["min"]
print(ranges.sort_values("range", ascending=False))
print(f"\nPossible total scores: {ranges['min'].sum()} to {ranges['max'].sum()}")

                             min  max  range
variable                                    
EXT_SOURCE_3                  19   69     50
EXT_SOURCE_2                  22   71     49
EXT_SOURCE_1                  36   62     26
AMT_GOODS_PRICE               38   55     17
DAYS_EMPLOYED                 42   55     13
ORGANIZATION_TYPE             40   53     13
CREDIT_TERM                   45   57     12
CREDIT_GOODS_RATIO            39   50     11
OWN_CAR_AGE                   44   53      9
REGION_RATING_CLIENT_W_CITY   43   52      9
NAME_EDUCATION_TYPE           45   53      8
OCCUPATION_TYPE               42   50      8

Possible total scores: 455 to 680


### Score every client, and check the scores against actual defaults

In [34]:
def to_score(X):
    logit = model.decision_function(X[FINAL_VARS])
    return offset - factor * logit

train["SCORE"] = to_score(X_train)
test["SCORE"] = to_score(X_test)

bands = pd.cut(test["SCORE"], bins=range(450, 700, 20))
summary = test.groupby(bands, observed=True)["TARGET"].agg(["count", "mean"])
summary["mean"] = summary["mean"].map("{:.1%}".format)
print(summary)

            count    mean
SCORE                    
(450, 470]      4  100.0%
(470, 490]    202   50.0%
(490, 510]   1589   36.6%
(510, 530]   6346   23.4%
(530, 550]  15656   13.6%
(550, 570]  24840    7.6%
(570, 590]  24360    3.7%
(590, 610]  14053    2.2%
(610, 630]   4503    1.2%
(630, 650]    670    0.7%
(650, 670]     31    0.0%


**Score bands on the test set:** the default rate falls steadily from 50% (scores 470–490) to 0.7% (630–650), and the odds roughly double every 20 points, as the PDO scaling intends. Clients scoring 590–610 default at 2.2%, close to the 2% implied by the base score of 600 at odds 50:1. The extreme bands contain very few clients (4 and 31) and are not interpreted.

## Save the scorecard

In [38]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")

joblib.dump({
    "model": model,
    "final_vars": FINAL_VARS,
    "factor": factor,
    "offset": offset,
    "scaling": {"pdo": PDO, "base_score": BASE_SCORE, "base_odds": BASE_ODDS},
}, MODEL_DIR / "pd_model.joblib")
scorecard.to_csv(MODEL_DIR / "scorecard.csv", index=False)

print(f"Saved PD model ({len(FINAL_VARS)} variables) and scorecard ({len(scorecard)} rows)")

Saved PD model (12 variables) and scorecard (93 rows)
